# add-sub-div-back-lambdas — worked example 2: unbroadcast a gradient back to the input shape

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `add-sub-div-back-lambdas`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a forward op broadcasts a small tensor against a big one, the backward must SUM the gradient over the broadcasted axes so the returned gradient matches the original input shape. `unbroadcast(g, x)` first sums away any leading axes that broadcasting added, then sums (keepdim) over any axis where `x` had size 1. This is the helper every add/sub/div backward lambda wraps around its raw partial.

## Worked solution

**Goal.** Implement `unbroadcast(grad, target)` so that a gradient computed at the broadcast (output) shape is reduced back to `target.shape`.

**Step 1 — drop extra leading dims.** Broadcasting can prepend axes: `target` of shape `(3,)` broadcast against `(2, 3)` gains a leading axis. So while `grad.ndim > target.ndim`, sum over axis 0. After this loop `grad.ndim == target.ndim`.

**Step 2 — collapse size-1 axes.** For any axis where `target` had size 1 but the output had size > 1, broadcasting copied the value; the backward must sum those copies. We loop over axes and `grad = grad.sum(dim=i, keepdim=True)` wherever `target.shape[i] == 1`. `keepdim=True` preserves the axis so the final shape equals `target.shape` exactly.

**Step 3 — why this is correct.** A broadcast copy means the same input element fed multiple output elements; gradients from parallel paths add, and summation over the copied axis is exactly that addition.

**Step 4 — verify.** We broadcast-add a `(1, 3)` tensor to a `(4, 3)` tensor, push a random upstream grad through, and confirm `unbroadcast` reproduces the autograd gradient (shape `(1, 3)`).

In [ ]:
def unbroadcast(grad, target):
    while grad.ndim > target.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(target.shape):
        if size == 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

t.manual_seed(0)
x = t.randn(1, 3, requires_grad=True)   # will broadcast over rows
y = t.randn(4, 3, requires_grad=True)
out = x + y                              # shape (4, 3)
g = t.randn(4, 3)

# add arg0 backward is just g, then unbroadcast back to x.shape
dx = unbroadcast(g, x.detach())

out.backward(g)
print('dx shape:', tuple(dx.shape))
print('dx match:', t.allclose(dx, x.grad, atol=1e-5))